# 06 - Initial Feature Engineering

Engineer predictive features from the cleaned match data: team/venue/toss encodings, historical win percentage, head-to-head win rate, venue win percentage, recent form, and (optionally) average venue score from ball-by-ball data.

**Leakage guard:** every historical stat (win %, form, head-to-head) is computed using only matches strictly *before* the one being featurized — never the match itself or future matches.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

MODELS_DIR = '../models'

matches = pd.read_csv('../data/processed/matches_cleaned.csv', parse_dates=['date'])
matches = matches[
    matches['winner'].notna()
    & ((matches['winner'] == matches['team1']) | (matches['winner'] == matches['team2']))
].copy()
matches = matches.sort_values('date').reset_index(drop=True)
matches['team1_won'] = (matches['winner'] == matches['team1']).astype(int)
matches.shape

(1187, 12)

## 1-5. Categorical encodings (team1, team2, venue, toss winner, toss decision)

Reuse the encoders fit in `05_preprocessing.ipynb` so encodings stay consistent across notebooks.

In [2]:
team_encoder = joblib.load(f'{MODELS_DIR}/team_encoder.joblib')
venue_encoder = joblib.load(f'{MODELS_DIR}/venue_encoder.joblib')
toss_decision_encoder = joblib.load(f'{MODELS_DIR}/toss_decision_encoder.joblib')

matches['team1_enc'] = team_encoder.transform(matches['team1'])
matches['team2_enc'] = team_encoder.transform(matches['team2'])
matches['toss_winner_enc'] = team_encoder.transform(matches['toss_winner'])
matches['venue_enc'] = venue_encoder.transform(matches['venue'])
matches['toss_decision_enc'] = toss_decision_encoder.transform(matches['toss_decision'])

## Long format helper

Historical stats (win %, form, head-to-head) are easiest to compute in a "long" table with one row per team-per-match, sorted chronologically, then merged back onto `team1`/`team2`.

In [3]:
team1_rows = matches[['id', 'date', 'venue', 'team1', 'team2', 'team1_won']].rename(
    columns={'team1': 'team', 'team2': 'opponent', 'team1_won': 'won'}
)
team2_rows = matches[['id', 'date', 'venue', 'team1', 'team2', 'team1_won']].rename(
    columns={'team2': 'team', 'team1': 'opponent'}
)
team2_rows['won'] = 1 - matches['team1_won']

long_df = pd.concat([team1_rows[['id', 'date', 'venue', 'team', 'opponent', 'won']],
                      team2_rows[['id', 'date', 'venue', 'team', 'opponent', 'won']]])
long_df = long_df.sort_values(['team', 'date']).reset_index(drop=True)
long_df['pair_key'] = long_df.apply(lambda r: '_'.join(sorted([r['team'], r['opponent']])), axis=1)
long_df.head()

,id,date,venue,team,opponent,won,pair_key
0,335983,2008-04-19,"Punjab Cricket Association Stadium, Mohali",Chennai Super Kings,Punjab Kings,1,Chennai Super Kings_Punjab Kings
1,335989,2008-04-23,"MA Chidambaram Stadium, Chepauk",Chennai Super Kings,Mumbai Indians,1,Chennai Super Kings_Mumbai Indians
2,335993,2008-04-26,"MA Chidambaram Stadium, Chepauk",Chennai Super Kings,Kolkata Knight Riders,1,Chennai Super Kings_Kolkata Knight Riders
3,335996,2008-04-28,M Chinnaswamy Stadium,Chennai Super Kings,Royal Challengers Bengaluru,1,Chennai Super Kings_Royal Challengers Bengaluru
4,336001,2008-05-02,"MA Chidambaram Stadium, Chepauk",Chennai Super Kings,Delhi Capitals,0,Chennai Super Kings_Delhi Capitals


## 6. Team historical win percentage

Expanding win rate for each team, shifted by one match so the current match is excluded. Teams with no prior matches default to a neutral 0.5.

In [4]:
long_df['team_win_pct'] = (
    long_df.groupby('team')['won']
    .apply(lambda s: s.shift().expanding().mean())
    .reset_index(level=0, drop=True)
    .fillna(0.5)
)

## 7. Head-to-head win rate

Each team's win rate against this specific opponent, based on prior meetings only.

In [5]:
long_df['h2h_win_rate'] = (
    long_df.groupby(['team', 'pair_key'])['won']
    .apply(lambda s: s.shift().expanding().mean())
    .reset_index(level=[0, 1], drop=True)
    .fillna(0.5)
)

## 8. Venue win percentage

Each team's historical win rate at this specific venue, prior meetings only.

In [6]:
long_df['venue_win_pct'] = (
    long_df.groupby(['team', 'venue'])['won']
    .apply(lambda s: s.shift().expanding().mean())
    .reset_index(level=[0, 1], drop=True)
    .fillna(0.5)
)

## 9. Recent form (last 5 matches)

In [7]:
long_df['recent_form_5'] = (
    long_df.groupby('team')['won']
    .apply(lambda s: s.shift().rolling(5, min_periods=1).mean())
    .reset_index(level=0, drop=True)
    .fillna(0.5)
)

## Merge engineered stats back onto matches (team1 and team2 sides)

In [8]:
stat_cols = ['team_win_pct', 'h2h_win_rate', 'venue_win_pct', 'recent_form_5']

team1_stats = long_df.merge(matches[['id', 'team1']], left_on=['id', 'team'], right_on=['id', 'team1'])
team2_stats = long_df.merge(matches[['id', 'team2']], left_on=['id', 'team'], right_on=['id', 'team2'])

features = matches.copy()
features = features.merge(
    team1_stats[['id'] + stat_cols].rename(columns={c: f'team1_{c}' for c in stat_cols}), on='id'
)
features = features.merge(
    team2_stats[['id'] + stat_cols].rename(columns={c: f'team2_{c}' for c in stat_cols}), on='id'
)
features.shape

(1187, 25)

## 10. Average venue score (optional, from deliveries.csv)

Skipped gracefully if `deliveries.csv` isn't available yet — the rest of the feature set doesn't depend on it.

In [9]:
deliveries_path = '../data/raw/deliveries.csv'

if os.path.exists(deliveries_path):
    deliveries = pd.read_csv(deliveries_path)
    innings_totals = deliveries.groupby(['match_id', 'innings'])['total_runs'].sum().reset_index()
    match_venue = matches[['id', 'venue']].rename(columns={'id': 'match_id'})
    innings_totals = innings_totals.merge(match_venue, on='match_id')
    avg_venue_score = innings_totals.groupby('venue')['total_runs'].mean().rename('avg_venue_score')
    features = features.merge(avg_venue_score, on='venue', how='left')
    features['avg_venue_score'] = features['avg_venue_score'].fillna(features['avg_venue_score'].mean())
else:
    print(f'{deliveries_path} not found — skipping avg_venue_score (place deliveries.csv in data/raw/ to enable it)')

/var/folders/3p/x4pf0wsj5sb9_jr6fwjvvrw40000gn/T/ipykernel_12056/3164207993.py:4: DtypeWarning: Columns (0: season) have mixed types. Specify dtype option on import or set low_memory=False.
  deliveries = pd.read_csv(deliveries_path)


## Save feature-engineered dataset

In [10]:
features.to_csv('../data/processed/matches_features.csv', index=False)
features.shape

(1187, 26)

## Feature summary table

In [11]:
feature_summary = pd.DataFrame([
    {'feature': 'team1_enc', 'description': 'Label-encoded team1 identity', 'source': 'team_encoder.joblib'},
    {'feature': 'team2_enc', 'description': 'Label-encoded team2 identity', 'source': 'team_encoder.joblib'},
    {'feature': 'venue_enc', 'description': 'Label-encoded venue', 'source': 'venue_encoder.joblib'},
    {'feature': 'toss_winner_enc', 'description': 'Label-encoded toss winner', 'source': 'team_encoder.joblib'},
    {'feature': 'toss_decision_enc', 'description': 'Label-encoded toss decision (bat/field)', 'source': 'toss_decision_encoder.joblib'},
    {'feature': 'team1_team_win_pct', 'description': "Team1's all-time win % before this match", 'source': 'engineered'},
    {'feature': 'team2_team_win_pct', 'description': "Team2's all-time win % before this match", 'source': 'engineered'},
    {'feature': 'team1_h2h_win_rate', 'description': "Team1's win rate vs team2 in prior meetings", 'source': 'engineered'},
    {'feature': 'team2_h2h_win_rate', 'description': "Team2's win rate vs team1 in prior meetings", 'source': 'engineered'},
    {'feature': 'team1_venue_win_pct', 'description': "Team1's win % at this venue historically", 'source': 'engineered'},
    {'feature': 'team2_venue_win_pct', 'description': "Team2's win % at this venue historically", 'source': 'engineered'},
    {'feature': 'team1_recent_form_5', 'description': "Team1's win rate over its last 5 matches", 'source': 'engineered'},
    {'feature': 'team2_recent_form_5', 'description': "Team2's win rate over its last 5 matches", 'source': 'engineered'},
    {'feature': 'avg_venue_score', 'description': 'Average innings total runs at this venue (optional, needs deliveries.csv)', 'source': 'deliveries.csv'},
    {'feature': 'season', 'description': 'IPL season year', 'source': 'raw'},
    {'feature': 'team1_won', 'description': 'Target: 1 if team1 won, else 0', 'source': 'derived from winner'},
])

feature_summary.to_csv('../data/processed/feature_summary.csv', index=False)
feature_summary

,feature,description,source
0,team1_enc,Label-encoded team1 identity,team_encoder.joblib
1,team2_enc,Label-encoded team2 identity,team_encoder.joblib
2,venue_enc,Label-encoded venue,venue_encoder.joblib
3,toss_winner_enc,Label-encoded toss winner,team_encoder.joblib
4,toss_decision_enc,Label-encoded toss decision (bat/field),toss_decision_encoder.joblib
5,team1_team_win_pct,Team1's all-time win % before this match,engineered
6,team2_team_win_pct,Team2's all-time win % before this match,engineered
7,team1_h2h_win_rate,Team1's win rate vs team2 in prior meetings,engineered
8,team2_h2h_win_rate,Team2's win rate vs team1 in prior meetings,engineered
9,team1_venue_win_pct,Team1's win % at this venue historically,engineered
